# 00: Look at the data

Before any model: what are the images, what are the human labels, and where would labeling be
hard? Run all cells, then look at the plots.

In [ ]:
# Setup: installs what is missing (Colab / RunPod) and finds vision.py
import importlib.util, pathlib, subprocess, sys, time, urllib.request
for pkg, mod in [("transformers", "transformers"), ("ultralytics", "ultralytics"),
                 ("segmentation-models-pytorch", "segmentation_models_pytorch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if (pathlib.Path.cwd().parent / "vision.py").exists():   # opened from vision/notebooks or vision/sol
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
elif not pathlib.Path("vision.py").exists():            # Colab: fetch it from the course repo
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhaw-physical-ai/cas-pair/main/vision/vision.py", "vision.py")
from vision import *
device = get_device()
print("device:", device)
if device == "cuda":   # YOLO needs torchvision's NMS on the GPU; some new GPUs (Blackwell) lack its code
    import torchvision
    try:
        torchvision.ops.nms(torch.tensor([[0., 0, 1, 1]], device="cuda"), torch.ones(1, device="cuda"), 0.5)
    except Exception as e:
        print("WARNING: YOLO will crash on this GPU:", e, "\nOn RunPod: deploy an RTX A4000, L4 or RTX 4090 instead (RUNPOD.md)")

In [ ]:
images, instances, train, test = load_pennfudan()      # ~50 MB on first use
n_people = [len(np.unique(m)) - 1 for m in instances]
sizes = np.array([im.shape[:2] for im in images])
print(f"{len(images)} images: {len(train)} train, {len(test)} test")
print(f"{sum(n_people)} people, {min(n_people)}-{max(n_people)} per image")
print(f"image height {sizes[:, 0].min()}-{sizes[:, 0].max()} px, width {sizes[:, 1].min()}-{sizes[:, 1].max()} px")

## Images are numbers
The network never sees a "person", only these numbers.

In [ ]:
im, inst = images[train[0]], instances[train[0]]
print("image:", im.shape, im.dtype, "values", im.min(), "-", im.max())
print("one pixel (R, G, B):", im[100, 100])
print("instance mask:", inst.shape, "ids", np.unique(inst), "(0 = background)")

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
show(axs[0], im, title="image")
axs[1].imshow(inst, cmap="tab10"); axs[1].set_title("instance mask: one id per person"); axs[1].axis("off")
show(axs[2], im, inst, boxes=boxes_from_instances(inst), title="mask + boxes")
plt.tight_layout()

## The human labels
The box is what an annotator draws in a few seconds. The mask takes about a minute per person.

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 9))
for ax, i in zip(axs.flat, train[1:9]):
    show(ax, images[i], instances[i], boxes=boxes_from_instances(instances[i]), title=f"image {i}")
plt.tight_layout()

## How big are the people?

In [ ]:
heights = [np.ptp(np.nonzero(m == pid)[0]) / m.shape[0] for m in instances for pid in np.unique(m) if pid]
plt.hist(heights, bins=20)
plt.xlabel("person height / image height"); plt.ylabel("# people")
plt.show()
print(f"on average {np.mean([(m > 0).mean() for m in instances]):.0%} of the pixels are person")

**✍️ Result and explanation:** Where would *you* hesitate if you had to draw these masks
(bags, hair, people at the edge or far away)? Which people will a model most likely miss?

> 